In [8]:
from kafka import KafkaConsumer
import json
import sys                                                                                            
sys.path.insert(0, '../')  

from src.models import Ride, ride_deserializer

In [6]:
server = 'localhost:9092'
topic_name = 'rides'

In [10]:
consumer = KafkaConsumer(
    topic_name,
    bootstrap_servers=[server],
    auto_offset_reset='earliest',
    group_id='rides-group',
    value_deserializer=ride_deserializer
)


In [19]:
next(consumer)

ConsumerRecord(topic='rides', partition=0, leader_epoch=2, offset=8, timestamp=1774463243262, timestamp_type=0, key=None, value=Ride(PULocationID=138, DOLocationID=37, trip_distance=8.4, total_amount=48.65, tpep_pickup_datetime=1761956330000), headers=[], checksum=None, serialized_key_size=-1, serialized_value_size=125, serialized_header_size=-1)

In [21]:
test_bytes = json.dumps({
    'PULocationID': 186,
    'DOLocationID': 79,
    'trip_distance': 1.72,
    'total_amount': 17.31,
    'tpep_pickup_datetime': 1730429702000
}).encode('utf-8')

ride_deserializer(test_bytes)
# Ride(PULocationID=186, DOLocationID=79, trip_distance=1.72,
#      total_amount=17.31, tpep_pickup_datetime=1730429702000)

Ride(PULocationID=186, DOLocationID=79, trip_distance=1.72, total_amount=17.31, tpep_pickup_datetime=1730429702000)

In [22]:
from datetime import datetime

print(f"Listening to {topic_name}...")

count = 0
for message in consumer:
    ride = message.value
    pickup_dt = datetime.fromtimestamp(ride.tpep_pickup_datetime / 1000)
    print(f"Received: PU={ride.PULocationID}, DO={ride.DOLocationID}, "
          f"distance={ride.trip_distance}, amount=${ride.total_amount:.2f}, "
          f"pickup={pickup_dt}")
    count += 1
    if count >= 10:
        print(f"\n... received {count} messages so far (stopping after 10 for demo)")
        break

consumer.close()

Listening to rides...
Received: PU=90, DO=100, distance=0.85, amount=$16.45, pickup=2025-10-31 17:21:11
Received: PU=142, DO=170, distance=3.01, amount=$25.85, pickup=2025-10-31 17:07:31
Received: PU=237, DO=144, distance=3.82, amount=$57.54, pickup=2025-10-31 17:46:52
Received: PU=162, DO=161, distance=0.89, amount=$12.95, pickup=2025-10-31 17:56:59
Received: PU=234, DO=162, distance=2.28, amount=$38.68, pickup=2025-10-31 17:10:43
Received: PU=158, DO=88, distance=3.3, amount=$44.00, pickup=2025-10-31 17:00:03
Received: PU=88, DO=148, distance=1.5, amount=$19.55, pickup=2025-10-31 17:43:53
Received: PU=148, DO=236, distance=4.7, amount=$47.65, pickup=2025-10-31 17:58:02
Received: PU=87, DO=255, distance=5.61, amount=$38.85, pickup=2025-10-31 17:52:48
Received: PU=231, DO=43, distance=3.9, amount=$46.55, pickup=2025-10-31 17:05:53

... received 10 messages so far (stopping after 10 for demo)
